In [16]:
#Importação das bibliotecas
import pandas as pd
import numpy as np
import io

In [17]:
url = "../../database/completo/df_treatment_part02.csv"
# url = "../../database/completo/df_completo_01_uf_br.csv"

# carrega dados
df = pd.read_csv( url, sep=",", encoding="UTF-8")

In [18]:
# Concatenação de uf e br

df["uf_br_km"] = df["uf"].astype(str) + "_" + df["br"].astype(str) + "_" + df["km"].astype(str)

df = df.drop(columns=['uf','br','km'])

In [19]:
# substituir -1 por 0
df["pista_molhada"] = df["pista_molhada"].replace(-1, 0)
df["visibilidade_ruim"] = df["visibilidade_ruim"].replace(-1, 0)

In [20]:
# categóricos binários

# Dicionário de mapeamento
mapa_sentido_via = {
    'Crescente': 0,
    'Decrescente': 1
}

# Aplicando transformação
df['sentido_via'] = df['sentido_via'].map(mapa_sentido_via)

# tratamento das condições meteorológicas

# Dicionário de mapeamento
mapa_uso_solo = {
    'Não': 0,
    'Sim': 1
}

# Aplicando transformação
df['uso_solo'] = df['uso_solo'].map(mapa_uso_solo)

In [21]:
# Categóricos múltiplos
# mapeamento tipo_pista

mapa_tipo_pista = {
  'Simples':0,
  'Dupla':1,
  'Múltipla': 2
}

# Aplicando transformação
df['tipo_pista'] = df['tipo_pista'].map(mapa_tipo_pista)

In [22]:
import numpy as np
import pandas as pd

def cyclic_encode(df, col, period, mapping=None, offset=0, prefix=None):
    """
    Aplica codificação cíclica (sin, cos) em uma coluna.

    Params:
    - df: DataFrame
    - col: nome da coluna
    - period: período do ciclo (ex: 7, 12, 4)
    - mapping: dict opcional para mapear categorias → inteiros
    - offset: ajuste (ex: mês começa em 1 → offset=1)
    - prefix: prefixo das novas colunas

    Retorna:
    - DataFrame com colunas sin/cos adicionadas
    """
    if prefix is None:
        prefix = col

    # mapear se necessário
    if mapping:
        values = df[col].map(mapping)
    else:
        values = df[col]

    # garantir numérico
    values = values.astype(float)

    # aplicar transformação
    angle = 2 * np.pi * (values - offset) / period
    df[f"{prefix}_sin"] = np.sin(angle)
    df[f"{prefix}_cos"] = np.cos(angle)

    return df

In [23]:
# Categóricos ciclícos
# mapeamento turno

mapa_turno = {
  'madrugada': 0,
  'manhã':1,
  'tarde': 2,
  'noite': 3
}

# Dicionário de mapeamento -> dia semana
mapa_dia_semana = {
    'segunda-feira': 0,
    'terça-feira': 1,
    'quarta-feira': 2,
    'quinta-feira': 3,
    'sexta-feira': 4,
    'sábado': 5,
    'domingo': 6,
}

# --- aplicar encoding ---
df = cyclic_encode(df, "turno", period=4, mapping=mapa_turno)
df = cyclic_encode(df, "dia_semana", period=7, mapping=mapa_dia_semana)
df = cyclic_encode(df, "mes", period=12, offset=1)  # mês começa em 1

# --- remover colunas originais ---
df = df.drop(columns=["turno", "dia_semana", "mes"])

In [24]:
df.shape

(632560, 22)

In [25]:
df.isnull().sum()

sentido_via           1403
pista_molhada            0
visibilidade_ruim        0
tipo_pista               0
uso_solo                 0
intersecao_de_vias       0
reta                     0
declive                  0
condicao_ignorada        0
feriado                  0
ano                      0
vel_max                  0
distancia_radar_m        0
frequencia            1403
grave                    0
uf_br_km                 0
turno_sin                0
turno_cos                0
dia_semana_sin           0
dia_semana_cos           0
mes_sin                  0
mes_cos                  0
dtype: int64

In [26]:
df = df.dropna()

In [27]:
df.shape

(631157, 22)

In [28]:
# Tratamento do nome dos atributos

import unidecode

# Supondo que df_agrupados já esteja carregado
# Exemplo de DataFrame

# Função para modificar os nomes das colunas
def formatar_nome(nome):
    nome = unidecode.unidecode(nome)  # Remove acentuação
    nome = nome.lower()  # Converte para minúsculas
    nome = nome.replace(' ', '_')  # Substitui espaços por underscore
    nome = nome.replace('-', '_')  # Substitui espaços por underscore
    return nome

# Aplica a formatação nas colunas do DataFrame
df.columns = [formatar_nome(col) for col in df.columns]

In [29]:
# --- Separação temporal ---
df_train = df[df["ano"].between(2017, 2024)].copy()
df_test  = df[df["ano"] == 2025].copy()

# --- Separação de X e y ---
X_train = df_train.drop(columns=["grave"]).copy()
y_train = df_train["grave"].copy()

X_test = df_test.drop(columns=["grave"]).copy()
y_test = df_test["grave"].copy()

# --- Target Encoding ---
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=["uf_br_km"], smoothing=10)

# fit no treino
X_train["uf_br_km"] = encoder.fit_transform(X_train["uf_br_km"], y_train)

# transform no teste
X_test["uf_br_km"] = encoder.transform(X_test["uf_br_km"])

# 🔹 Atualizar os DataFrames completos
df_train["uf_br_km"] = X_train["uf_br_km"]
df_test["uf_br_km"]  = X_test["uf_br_km"]

In [30]:
df_test.shape

(72310, 22)

In [31]:
df_train.shape

(558847, 22)

In [33]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

target_encoder = [
    'uf_br'
]

ciclicas = [
    'turno_sin', 'turno_cos',
    'dia_semana_sin', 'dia_semana_cos',
    'mes_sin', 'mes_cos'
]

numericas = [
    'vel_max',
    'distancia_radar_m',
    'frequencia'
]

binarias = [
    'sentido_via',
    'pista_molhada',
    'visibilidade_ruim',
    'tipo_pista',
    'uso_solo',
    'intersecao_de_vias',
    'reta',
    'declive',
    'condicao_ignorada',
    'feriado'
]
df_train[numericas] = scaler.fit_transform(df_train[numericas])
df_test[numericas] = scaler.fit_transform(df_test[numericas])

In [34]:
df_train.isnull().sum()

sentido_via           0
pista_molhada         0
visibilidade_ruim     0
tipo_pista            0
uso_solo              0
intersecao_de_vias    0
reta                  0
declive               0
condicao_ignorada     0
feriado               0
ano                   0
vel_max               0
distancia_radar_m     0
frequencia            0
grave                 0
uf_br_km              0
turno_sin             0
turno_cos             0
dia_semana_sin        0
dia_semana_cos        0
mes_sin               0
mes_cos               0
dtype: int64

In [35]:
# Salvando como CSV, sem o índice (index=False)
df_train.to_csv('../../database/completo/final/df_train.csv', index=False, encoding='utf-8')
df_test.to_csv('../../database/completo/final/df_test.csv', index=False, encoding='utf-8')

In [36]:
df.describe()

,sentido_via,pista_molhada,visibilidade_ruim,tipo_pista,uso_solo,intersecao_de_vias,reta,declive,condicao_ignorada,feriado,...,vel_max,distancia_radar_m,frequencia,grave,turno_sin,turno_cos,dia_semana_sin,dia_semana_cos,mes_sin,mes_cos
count,631157.000000,631157.000000,631157.000000,631157.000000,631157.000000,631157.000000,631157.000000,631157.000000,631157.000000,631157.000000,...,631157.000000,6.311570e+05,631157.000000,631157.000000,6.311570e+05,6.311570e+05,631157.000000,631157.000000,631157.000000,6.311570e+05
mean,0.465190,0.146152,0.154485,0.586737,0.432414,0.060996,0.681616,0.079611,0.014085,-0.327050,...,92.278498,3.326697e+05,29.431864,0.130945,-7.966322e-03,-1.795892e-01,-0.080866,-0.005054,-0.018911,9.606339e-04
std,0.498787,0.353259,0.361413,0.642682,0.495411,0.239323,0.465850,0.270690,0.117843,24.354415,...,24.600266,5.181279e+05,51.398845,0.337341,7.525315e-01,6.335473e-01,0.706816,0.702743,0.705165,7.087913e-01
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-64.000000,...,40.000000,5.600000e-01,1.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,-0.900969,-1.000000,-1.000000e+00
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-13.000000,...,90.000000,4.220620e+03,3.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.781831,-0.900969,-0.866025,-8.660254e-01
50%,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,100.000000,3.729767e+04,8.000000,0.000000,1.224647e-16,-1.836970e-16,0.000000,-0.222521,0.000000,-1.836970e-16
75%,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,12.000000,...,110.000000,5.133690e+05,31.000000,0.000000,1.000000e+00,6.123234e-17,0.433884,0.623490,0.500000,8.660254e-01
max,1.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,64.000000,...,110.000000,2.368404e+06,474.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,1.000000,1.000000,1.000000e+00
